In [1]:
# Standard library imports
import concurrent
import json
import os
import re
import shutil
from pathlib import Path

# Third-party imports
import chromadb
import cv2
import fitz  # PyMuPDF
import layoutparser as lp
import matplotlib.pyplot as plt
import numpy as np
import ollama
import pandas as pd
from PIL import Image
from tqdm import tqdm  # (or use 'from tqdm.auto import tqdm' if running in a Jupyter Notebook)

# LangChain imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings

# Commented out / unused imports from your original script
# from surya.layout import LayoutPredictor
# from langchain_community.embeddings import OllamaEmbeddings

In [2]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [ ]:
EXTRACTED_DIR = "extracted_data"
IMAGES_DIR = os.path.join(EXTRACTED_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)


In [ ]:
FIGURE_LABELS = {"Picture", "Figure", "Image", "Fig"}
TABLE_LABELS  = {"Table"}
TARGET_LABELS = FIGURE_LABELS | TABLE_LABELS


In [ ]:
def find_cap(text_blocks, bbox, box_type):
    x1, y1, x2, y2 = bbox 
    best, min_dist = "", float("inf")

    for b in text_blocks:
        # We no longer need the b[6] check because you already 
        # filtered non-text blocks in the main loop!
        
        # Unpack from your custom dictionary keys
        tx0, ty0, tx1, ty1 = b['bbox']
        text = b['text']

        horiz_overlap = max(0, min(x2, tx1) - max(x1, tx0))
        if horiz_overlap < 10 and (x2 - x1) > 100:
            continue

        if box_type == "Figure":
            dist = ty0 - y2  # distance below the figure
            if -20 < dist < 400:
                score = dist - (200 if text.lower().startswith("fig") else 0)
                if score < min_dist:
                    min_dist, best = score, text

    return best

In [ ]:
def clean_text(text):
    # Collapse spaced-out characters: "W e  w i l l" -> "We will"
    text = re.sub(r'\b(\w) (\w) ', r'\1\2', text)  # iterative collapse
    # Run it multiple times since each pass only collapses pairs
    for _ in range(20):
        text = re.sub(r'(?<!\w)(\w) (\w)(?!\w)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [ ]:
query = "Motivated by recent advances on local conductance measurement techniques at the nanoscale, timely questions are being raised about what possible information can be extracted from a disordered graphene sheet by selectively interrogating its transport properties. Here we demonstrate how an inversion technique originally developed to identify the number of scatterers in a quantum device can be adapted to a multi-terminal setup in order to provide detailed information about the spatial distribution of impurities on the surface of graphene, as well as other 2D material systems."

In [ ]:
clean_text(query)

In [ ]:
detecteon_weights = os.path.abspath("model_final.pth")
model = lp.Detectron2LayoutModel(
        config_path='lp://PubLayNet/mask_rcnn_X_101_32x8d_FPN_3x/config',
        model_path=detecteon_weights,
        extra_config=["MODEL.ROI_HEADS.SCORE_THRESH_TEST", 0.5], # 50% confidence threshold
        label_map={0: "Text", 1: "Title", 2: "List", 3: "Table", 4: "Figure"}
    )


corpus = []

dpi = 200
zoom = dpi / 72.0


In [ ]:
for doc in pdfs:

    pdf = fitz.open(doc)
    doc_name = doc.stem
    pdf_name = doc_name.strip().replace(" ","_").lower()
    print(doc_name)
        # Wrap the range in tqdm to generate the progress bar
    for page_idx in tqdm(range(len(pdf)), desc="Processing PDF Pages", unit="page"):
        
        page = pdf[page_idx]

        pix = page.get_pixmap(dpi=dpi)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        layout = model.detect(img)

        raw_blocks = page.get_text("blocks")
        text_blovk_img = []

        for b in raw_blocks:
            if b[6] == 0:
                tx1, ty1, tx2, ty2 = [c * zoom for c in b[:4]]
                text = b[4].replace('\n', ' ')

                text_blovk_img.append({
                    'bbox': (tx1, ty1, tx2, ty2),
                    'text': text.strip()
                })
                
                if len(text) > 4:
                # Add text as a structured dictionary to the corpus
                    corpus.append({
                        "document": pdf_name,
                        "page": page_idx,
                        "type": "text",
                        "content": text
                    })

        figures = [b for b in layout if b.type in ["Figure", "Table"]]

        for i, fig in enumerate(figures):
            pad = 20
            x1, y1, x2, y2 = fig.coordinates
            x1 = max(0, int(x1 - pad))
            y1 = max(0, int(y1 - pad))
            x2 = min(img.shape[1], int(x2 + pad))
            y2 = min(img.shape[0], int(y2 + pad))
                        
            # Crop the image array
            cropped_img = img[y1:y2, x1:x2]
            
            # Save the cropped figure to disk
            out_name = f"{pdf_name}_p{page_idx}_f{i}.png"
            out_path = os.path.join("extracted_data/images/", out_name)
            cv2.imwrite(out_path, cropped_img)
                        
            # Attempt to find the text caption for this specific figure
            context = find_cap(text_blovk_img, (x1, y1, x2, y2), fig.type)
                
            # Save all properties so we can index them later
          #  all_metadata.append({
          #      "file_name" : pdf_name,
          #      "image_path": out_name,
          #      "context": context,
          #      "page": page_idx,
          #      "type": fig.type
          #  })

            try:
                formatted_prompt = f"You are analysing scientific plots. Describe this {fig.type.lower()}. Extract textual information, data and trends.\n\nSurrounding Document Context:\n{context}. Answer in 3-5 sentences at max.  "
                #print(formatted_prompt)
                response = ollama.chat(
                    model="gemma4:latest",
                    messages=[{
                        'role': 'user',
                        'content': formatted_prompt,
                        'images': [out_path] 
                    }]
                )

                vlm_description = response['message']['content']
                #print(vlm_description)
            except Exception as e:
                # We use tqdm.write() instead of print() so it doesn't break the progress bar visually
                tqdm.write(f"Ollama failed on {out_name}: {e}")
                vlm_description = "Description generation failed."

            corpus.append({
            "document": pdf_name,
            "page": page_idx,
            "type": fig.type.lower(),
            "content": vlm_description,
            "metadata": {
                "image_path": out_name,
                "caption": context
                }
            })


    output_file = f"extracted_data/{pdf_name}_corpus.jsonl"
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, 'w') as f:
        for entry in corpus:
            f.write(json.dumps(entry) + '\n')

    print(f"\nCorpus successfully built and saved to {output_file}")

In [3]:
total_corpus = []

json_files = list(Path("extracted_data/").glob("*_corpus.jsonl"))

In [4]:
print("Initializing Semantic Chunker...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

chunker = SemanticChunker(
    embeddings,
    # "percentile" means it will split when the semantic difference 
    # between sentences is in the top 10% of all differences in the document.
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=90 
)

Initializing Semantic Chunker...


In [ ]:
with open("extracted_data/p1_corpus.jsonl") as f:
    for line in f:
        entry = json.loads(line)
        print(entry)

In [ ]:
for file in json_files:
    print(file)
    try:
        with open(file, 'r') as f:
            for line in f:
                entry = json.loads(line)

                if entry['type'] in ["figure","table"]:
                    # For figures and tables, we keep the VLM-generated description as is
                    total_corpus.append(entry)
                    continue

                if entry['type'] == "text":
                    content = entry['content']

                    docs = chunker.create_documents([content])

                    for i, doc in enumerate(docs):
                        total_corpus.append({
                            "document": entry['document'],
                            "page": entry['page'],
                            "type": "text_chunk",
                            "content": doc.page_content,
                            "metadata": {
                                "original_text": content,
                                "chunk_index": i
                            }
                        })
    except Exception as e:
        print(f"Could not process file {file}: {e}")


with open("final_corpus.jsonl", 'w') as f:
    for entry in total_corpus:
        f.write(json.dumps(entry) + '\n')


print(f"Final corpus created with {len(total_corpus)} entries and saved to final_corpus.jsonl")

extracted_data/p2_corpus.jsonl
extracted_data/p3_corpus.jsonl
extracted_data/effective-medium-network_corpus.jsonl
extracted_data/nature13831_corpus.jsonl
extracted_data/1806-1117-rbef-39-01-e1303_corpus.jsonl
extracted_data/nature17151_corpus.jsonl
extracted_data/empirical_model_for_ratio_of_conductivity_corpus.jsonl


In [ ]:


db_path = "./physics_vectordb"

# Check if the old database exists, and delete it if it does
if os.path.exists(db_path):
    print("Removing old database...")
    shutil.rmtree(db_path)
    print("Old database removed!")

db_path_2 = "./chromadb"
if os.path.exists(db_path_2):
    print("Removing old chromadb directory...")
    shutil.rmtree(db_path_2)
    print("Old chromadb directory removed!")


# 1. Initialize the Database and Embedding Model
print("Initializing ChromaDB and Ollama...")
# Ensure you use the exact same embedding model you used for chunking
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# This creates a folder in your directory to save the database persistently
chroma_client = chromadb.PersistentClient(path="./physics_vectordb")

# Create a collection (like a table in SQL) using cosine similarity
collection = chroma_client.get_or_create_collection(
    name="physics_papers",
    metadata={"hnsw:space": "cosine"} 
)

input_file = "final_corpus.jsonl"

documents = []
metadatas = []
ids = []
# Create a set to track unique strings
seen_texts = set()

print("Reading and deduplicating chunked corpus...")
with open(input_file, 'r') as f:
    for line in f:
        entry = json.loads(line)
        content = entry["content"].strip()
        
        # Skip this chunk if we've already seen this exact text
        if content in seen_texts:
            continue
            
        # Add to our tracker
        seen_texts.add(content)
        
        documents.append(content)
        
        meta = {
            "document": entry["document"],
            "page": entry["page"],
            "type": entry["type"]
        }
        
        if "metadata" in entry:
            for k, v in entry["metadata"].items():
                meta[f"extra_{k}"] = str(v) 
                
        metadatas.append(meta)
        current_index = len(documents) - 1
        ids.append(f"chunk_{current_index}")

print(f"Filtered down to {len(documents)} unique chunks.")
# 3. Generate Embeddings and Load into ChromaDB
print(f"Generating embeddings for {len(documents)} chunks (this may take a moment)...")

# We process in batches to prevent memory overloads on your GPU/RAM
batch_size = 5000
for i in tqdm(range(0, len(documents), batch_size), desc="Ingesting Batches"):
    batch_docs = documents[i:i + batch_size]
    batch_meta = metadatas[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    
    # Generate the vector embeddings via Ollama
    batch_embeddings = embeddings.embed_documents(batch_docs)
    
    # Insert into the database
    collection.add(
        embeddings=batch_embeddings,
        documents=batch_docs,
        metadatas=batch_meta,
        ids=batch_ids
    )

print("\nSuccess! Vector database is fully loaded and saved to disk.")

In [ ]:


# Reconnect to the database
chroma_client = chromadb.PersistentClient(path="./physics_vectordb")
collection = chroma_client.get_collection(name="physics_papers")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Write a question related to the physics paper you just processed
query_text = "What is the misfit function?"

print(f"Searching for: '{query_text}'\n")

# Generate the embedding for the user's question
query_embedding = embeddings.embed_query(query_text)

# Retrieve the top 3 most semantically similar chunks
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

# Display the results
for i in range(len(results['documents'][0])):
    doc_text = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    distance = results['distances'][0][i] # Lower distance = higher similarity
    chunk_id = results['ids'][0][i]

    
    print(f"--- Result {i+1} (Score: {distance:.4f}) ---")
    print(f"Type: {meta['type'].upper()} | Page: {meta['page']}")
    print(f"ID: {chunk_id} | Page: {meta['page']}")
    # If the result is a figure, point out the image path!
    if meta['type'] in ['figure', 'table']:
        print(f"Image Path: {meta.get('extra_image_path', 'N/A')}")
        
    print(f"Content: {doc_text}\n")

In [ ]:
import chromadb
from rank_bm25 import BM25Okapi
import pickle
import re

print("Connecting to database...")
# 1. Connect to ChromaDB (The Single Source of Truth)
chroma_client = chromadb.PersistentClient(path="./physics_vectordb")
collection = chroma_client.get_collection(name="physics_papers")

# Pull only the documents and IDs
all_data = collection.get(include=["documents"])
metadatas = collection.get(include=["metadatas"])["metadatas"]
print(f"Fetched {len(all_data['documents'])} chunks. Aligning indices...")

# 2. Sort documents by their exact chunk ID to ensure perfect alignment
paired_data = []
for doc, chunk_id in zip(all_data["documents"], all_data["ids"]):
    # Extract the integer from "chunk_15"
    idx = int(chunk_id.split("_")[1])
    paired_data.append((idx, doc))

# Sort the list by the numerical index so text[0] is guaranteed to be chunk_0
paired_data.sort(key=lambda x: x[0])
texts = [item[1] for item in paired_data]

# 3. Robust Tokenization
print("Tokenizing corpus...")
tokenized_corpus = [re.findall(r'\w+', doc.lower()) for doc in texts]

# 4. Build and Pickle
print("Building BM25 Index...")
bm25 = BM25Okapi(tokenized_corpus)

with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)
    
print("Success! BM25 index saved to bm25_index.pkl")

In [ ]:
"""example"""

query = "universal conductance"
tokenized_query = query.lower().split(" ")
scores = bm25.get_scores(tokenized_query)
top_10 = np.argsort(scores)[::-1][:10]


In [ ]:
len(query)

In [ ]:
for i , inx in enumerate(top_10):
    if scores[inx] >12:
        print(scores[inx])
        dic = metadatas[inx]
        print(dic['source'])

In [ ]:
import re

def hybrid_search(query, top_k, rrf_k=60):
    k_cand = max(15, top_k * 3)

    # --- 1. Sparse (BM25) retrieval ---
    # Better tokenization: strips punctuation, keeps alphanumeric
    tokenized_query = re.findall(r'\w+', query.lower())
    bm25_scores = bm25.get_scores(tokenized_query)
    
    # Get top sparse indices ordered by score (highest first)
    sparse_top_indices = np.argsort(bm25_scores)[::-1][:k_cand]

    # --- 2. Dense (embedding) retrieval ---
    embeddings_model = OllamaEmbeddings(
        model="nomic-embed-text", 
        base_url="http://localhost:11434"
    )
    query_emb = embeddings_model.embed_query(query)

    dense_results = collection.query(
        query_embeddings=[query_emb], 
        n_results=k_cand,
        include=["documents", "metadatas", "distances"]
    )

    # Get top dense indices ordered by distance (lowest distance first)
    dense_ids_ordered = [int(id.split("_")[1]) for id in dense_results["ids"][0]]
    dense_distances = dense_results["distances"][0]

    # --- 3. Reciprocal Rank Fusion (RRF) ---
    # Formula: RRF_Score = 1 / (rank + k)
    fused_scores = {}

    # Add sparse ranks
    for rank, idx in enumerate(sparse_top_indices):
        fused_scores[idx] = fused_scores.get(idx, 0.0) + (1.0 / (rank + 1 + rrf_k))

    # Add dense ranks
    for rank, idx in enumerate(dense_ids_ordered):
        fused_scores[idx] = fused_scores.get(idx, 0.0) + (1.0 / (rank + 1 + rrf_k))

    # --- 4. Rank and return top_k ---
    # Sort by the combined RRF score
    ranked_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, rrf_score in ranked_results:
        # Optional: Grab the original scores for debugging/logging
        orig_bm25 = bm25_scores[idx]
        
        # Find the dense distance if it was in the dense retrieved set
        orig_dense_dist = None
        if idx in dense_ids_ordered:
            list_idx = dense_ids_ordered.index(idx)
            orig_dense_dist = dense_distances[list_idx]

        results.append({
            "chunk_index": idx,
            "text": texts[idx],
            "metadata": metadatas[idx],
            "rrf_score": round(rrf_score, 5),
            "raw_bm25": round(orig_bm25, 4),
            "raw_dense_distance": round(orig_dense_dist, 4) if orig_dense_dist is not None else "N/A"
        })

    return results

In [ ]:
results = hybrid_search("nanowire network", top_k=3, rrf_k=60)
for r in results:
    print(f"[{r['rrf_score']}] {r['text']}...")


In [ ]:
!ollama ls

In [ ]:
import ollama

def rag_answer(query, top_k=10):
    # 1. Retrieve context via hybrid search
    results = hybrid_search(query, top_k, rrf_k=60)

    # 2. Build context block from retrieved chunks
    context_block = ""
    for i, r in enumerate(results):
        source = r["metadata"].get("source", "unknown")
        context_block += f"--- Chunk {i+1} [source: {source}] ---\n"
        context_block += r["text"] + "\n\n"
    print(context_block)
    # 3. Prompt the LLM
    system_prompt = (
        "You are an expert scientific AI assistant. "
        "Answer the user's question using ONLY the provided context. "
        "Cite which chunk(s) you drew each part of the answer from. "
        "Generate a figure to describe the result where applicable"
        "look for mathematical descriptions"
        "If the context does not contain enough information, say so clearly."
    )

    user_prompt = (
        f"## Context\n{context_block}\n"
        f"## Question\n{query}\n\n"
        "Provide a detailed, well-structured answer."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    # 4. Stream the response
    print(f"Query: {query}\n{'='*60}")
    full_response = ""
    for chunk in ollama.chat(model="gemma4:latest", messages=messages, stream=True):
        token = chunk["message"]["content"]
        full_response += token
        print(token, end="", flush=True)
    
    print(f"\n{'='*60}")
    return full_response, results


In [ ]:
answer, sources = rag_answer("what is misfit function?")

In [ ]:
answer